[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Connection Pools &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with `terminate` and `backends`. Run it first. Each task
opens the pool it needs and closes it, so they can be run in any order.


In [1]:
import asyncio
import getpass
import json
import logging
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
import psycopg_pool
from asyncpg import exceptions

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

logging.getLogger("psycopg.pool").setLevel(logging.CRITICAL)        # its retries are not the lesson


def terminate(application_name):
    """Close every backend wearing one application_name, which is a server restart in miniature."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        return conn.execute("SELECT count(pg_terminate_backend(pid)) FROM pg_stat_activity "
                            "WHERE application_name = %s", (application_name,)).fetchone()[0]


def backends(application_name):
    """How many connections the server currently has under that name."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        return conn.execute("SELECT count(*) FROM pg_stat_activity WHERE application_name = %s",
                            (application_name,)).fetchone()[0]


def sessions():
    """How many connections the server has accepted since it started, counting from PostgreSQL 14."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:                # this one included
        return conn.execute("SELECT sessions FROM pg_stat_database "
                            "WHERE datname = 'guide'").fetchone()[0]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


**1.** A pool, one query, and a close.


In [2]:
pool = psycopg_pool.ConnectionPool("dbname=guide", min_size=2, max_size=2, open=True)
pool.wait(timeout=10)

with pool.connection() as conn:
    print("through the pool:", conn.execute("SELECT count(*) FROM events").fetchone())

pool.close()
print("closed")


through the pool: (5000,)
closed


`wait` is the line worth keeping. Without it the pool opens in the background and a server that is
not there is discovered by whichever request happens to be first.


**2.** Before and after wait.


In [3]:
pool = psycopg_pool.ConnectionPool("dbname=guide application_name=counted",
                                   min_size=3, max_size=3, open=True)
pool.wait(timeout=10)
print("after wait: ", backends("counted"))
pool.close()
print("after close:", backends("counted"))


after wait:  3
after close: 0


`open=True` starts the opening in the background, so asking before `wait` returns whatever number of
connections happened to be ready at that instant, which is not something to build on. After `wait`
it is `min_size`, every time. After `close` it is zero, because closing a pool closes what it holds.


**3.** Eight slow queries, two connections.


In [4]:
pool = await asyncpg.create_pool(database="guide", min_size=2, max_size=2)

async def slow(number):
    return await pool.fetchval("SELECT pg_sleep(0.2), $1::int", number)

start = time.perf_counter()
await asyncio.gather(*(slow(number) for number in range(8)))
print(f"8 queries of 0.2s through 2 connections: {time.perf_counter() - start:.1f}s")

await pool.close()


8 queries of 0.2s through 2 connections: 0.8s


Four turns of two. Eight tasks and two connections is eight tenths of a second, and the only way to
make it shorter is more connections, not more tasks.


**4.** One connection, not given back.


In [5]:
pool = psycopg_pool.ConnectionPool("dbname=guide", min_size=1, max_size=1, timeout=2, open=True)
pool.wait(timeout=10)

held = pool.getconn()
try:
    pool.getconn()
except psycopg_pool.PoolTimeout as error:
    print("PoolTimeout:", error)

pool.putconn(held)
print("after giving it back:", pool.getconn() is not None)
pool.close()


PoolTimeout: couldn't get a connection after 2.00 sec
after giving it back: True


Two seconds of waiting and then a clear exception. The message names the wait, not the cause, so the
place to look is always for a borrow that has no matching return.


**5.** A checked pool, after the server closes everything.


In [6]:
pool = psycopg_pool.ConnectionPool("dbname=guide application_name=survivor",
                                   min_size=2, max_size=2, open=True,
                                   check=psycopg_pool.ConnectionPool.check_connection)
pool.wait(timeout=10)

print("terminated", terminate("survivor"), "connections")
time.sleep(0.3)

with pool.connection() as conn:
    print("the next request still works:", conn.execute("SELECT 1").fetchone())
print("the pool has", backends("survivor"), "connections again")
pool.close()


terminated 2 connections
the next request still works: (1,)
the pool has 2 connections again


The same pool without `check` would have handed out a dead connection and raised. One round trip per
borrow buys the difference.


**6.** An init that names the connections.


In [7]:
async def name_it(conn):
    await conn.execute("SET application_name = 'from_init'")


pool = await asyncpg.create_pool(database="guide", min_size=3, max_size=3, init=name_it)
print("backends wearing that name:", backends("from_init"))

await pool.close()
print("after closing the pool:   ", backends("from_init"))


backends wearing that name: 3
after closing the pool:    0


`init` ran three times, once per connection the pool opened, which is the same hook that registered
the `jsonb` codec in the notebook. Anything a connection needs before your code touches it belongs
there.


---

&#8592; **Back to:** [Connection Pools](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/13-connection-pools.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
